# 06 — A toy global workspace

**Background resource:** Section numbers below refer to the [August perspective](../../paper/background/august-perspective.pdf), preserved alongside the current paper.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arvidl/brains-to-models-publication/blob/main/code/notebooks/06_global_workspace.ipynb)

Companion to §7 of *From Brains to Models*. Notebook 05 made “basin” something you can point at. This notebook is the next grain: a **limited-capacity broadcast**. Eight specialised modules run in parallel. A workspace of two slots opens only when some module crosses a threshold; the winners are written back to everyone. Below threshold the same computation stays local. That is Dehaene / Baars as a schematic — ignition versus local, unbroadcast processing.

It is not a cortex, not $\Phi$, and not Claude. Integrated information theory is named in the paper and not implemented here.

The fourth panel is a *map* of a different claim. Gurnee, Sofroniew, Pearce, Lindsey and colleagues (Transformer Circuits, 2026) report that language models maintain a small, privileged set of **verbalizable** representations — a J-space — atop a much larger volume of automatic processing. They take no position on phenomenal consciousness, and they deny that a transformer reproduces the brain’s workspace architecture. Panel D is a drawing of that split, not a measurement of Claude.

**Claim.** Most computation is not broadcast; crossing a threshold is a different event from being slightly more active; verbalizability is not phenomenal experience.

**Method.** Eight modules, two workspace slots, a threshold. Seed 7. Panel D is a schematic of Gurnee et al., not a measurement.

**Result.** Subthreshold drive decays locally; suprathreshold drive is held and broadcast; occupancy is a step, not a slope.

**Limit.** Not a cortex, not $\Phi$, not Claude, not evidence of experience.

<details>
<summary><b>Key terms — for clinicians and neuroscientists</b></summary>

- **[Global workspace theory](https://en.wikipedia.org/wiki/Global_workspace_theory).** Baars's and Dehaene's picture: many specialised processors run in parallel; occasionally one coalition wins and its content is *broadcast* to the rest. Broadcast content is what can be reported and flexibly used.
- **Ignition and broadcast.** The nonlinear event when a signal crosses threshold and is shared widely, as opposed to staying local. It is a step, not a gradual rise.
- **Limited capacity.** Only a little can be broadcast at once (here, two slots), which is why conscious access feels serial and narrow.
- **[Access versus phenomenal](https://plato.stanford.edu/entries/consciousness/) consciousness.** *Access* is information poised for report and control; *phenomenal* is what it is like to undergo an experience. This toy models access only.
- **Verbalizable representations (J-space).** A reported finding that language models keep a small privileged set of features that can be named and steered, atop much larger automatic processing — an access-like split, not evidence of experience.

</details>

<details>
<summary><b>Key terms — for the mathematically inclined</b></summary>

- **Modules and workspace.** $M=8$ modules produce activations; a workspace of $K=2$ slots admits a module only when its drive exceeds a threshold $\theta$.
- **Threshold nonlinearity.** Occupancy is a step function of drive: below $\theta$, local decay; above $\theta$, held and written back to all modules.
- **Broadcast.** The winners' contents are copied to every module — the "global" write.
- **Metric.** Workspace occupancy against input drive — a step, not a slope.
- **Not $\Phi$.** No integrated-information ([IIT](https://en.wikipedia.org/wiki/Integrated_information_theory)) computation and no perturbational complexity; access dynamics only. Seed 7.

</details>

Writes `paper/figures/global_workspace.pdf` (`make figures`).

Reproducible: `SEED = 7` in the setup cell. Restart and run all to reproduce the figure.

### Concepts used in this notebook

This notebook runs eight modules competing for a two-slot workspace with a threshold, and (panel D) sketches the verbalizable-representation claim for language models. The two blocks below define the recurring terms once, first in plain language and then in notation. Both cover the same concepts at a different level of detail.

<details>
<summary><b>Concepts — for clinicians and neuroscientists</b></summary>

- **Specialised modules.** Parallel processors (vision, language, memory, and so on) that mostly work locally and unconsciously.
- **Coalition and competition.** Modules compete; a winning coalition is what gets broadcast. Losing does not mean idle — it means unbroadcast.
- **Threshold (step versus slope).** Being slightly more active is not the same as crossing threshold; only the crossing triggers broadcast. The figure shows a step.
- **Subthreshold versus suprathreshold.** Below threshold, drive decays locally; above, it is sustained and shared. The same computation, two fates.
- **Verbalizability is not experience.** A model trained on self-report can produce fluent introspective text; that is access-like organisation, not evidence of what-it-is-like.
- **What this is and is not.** A schematic of Dehaene/Baars access and of Gurnee et al.'s J-space — not a cortex, not $\Phi$, not Claude, and not evidence of experience.

</details>

<details>
<summary><b>Concepts — for the mathematically inclined</b></summary>

- **Architecture.** $M=8$ modules, workspace capacity $K=2$, threshold $\theta$; seed 7.
- **Dynamics.** Module drive $d_i(t)$; admit to the workspace iff $d_i>\theta$; broadcast writes winners back to all modules; subthreshold drive decays.
- **Read-out.** Occupancy as a function of input drive — a step at $\theta$.
- **Panel D.** A drawing of the reported J-space split (a small verbalizable set atop a large automatic volume), not a measurement.
- **Theories named, not run.** [Global workspace theory](https://en.wikipedia.org/wiki/Global_workspace_theory) is implemented as a schematic; [IIT](https://en.wikipedia.org/wiki/Integrated_information_theory) / $\Phi$ is named only.
- **Limit.** Access, not [phenomenal experience](https://plato.stanford.edu/entries/consciousness/); the step is not a jump in what-it-is-like.

</details>

In [1]:
import os
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np

REPO = os.environ.get("BRAINSMODELS_REPO_URL",
                      "https://github.com/arvidl/brains-to-models-publication.git")
COLAB_ROOT = Path("/content/brains-to-models")

if "google.colab" in sys.modules:
    if not (COLAB_ROOT / "code" / "src" / "brainsmodels").is_dir():
        subprocess.check_call(["git", "clone", "--depth", "1", REPO, str(COLAB_ROOT)])
    root = COLAB_ROOT
else:
    here = Path.cwd().resolve()
    root = here.parents[1] if here.name == "notebooks" else here

src = root / "code" / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

from brainsmodels.repro import DEFAULT_SEED, set_seed
from brainsmodels.workspace import (
    CAPACITY,
    MODULE_NAMES,
    N_MODULES,
    THRESHOLD,
    facts,
    render_paper_figure,
    run_experiment,
    write_figure,
)

SEED = set_seed(DEFAULT_SEED)
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})
print("repo root:", root)
print("seed:", SEED)
print(f"{N_MODULES} modules, {CAPACITY} slots, θ = {THRESHOLD}")

repo root: <checkout>
seed: 7
8 modules, 2 slots, θ = 0.55


## The schematic

Eight modules $m=1,\ldots,8$ carry a scalar activity $s_m\in[0,1]$. A workspace of $k=2$ slots is empty unless someone has crossed a threshold $\theta$:

$$
\mathrm{occupy}(s)=\begin{cases}
\text{the }k\text{ strongest }m\text{ with }s_m\ge\theta & \text{if any such }m\text{ exist,}\\
\emptyset & \text{otherwise.}
\end{cases}
$$

Each step is local decay, an external drive, and a broadcast bonus for whoever is in:

$$
s\leftarrow\mathrm{clip}\bigl((1-\delta)\,s+d_t+\beta\,\mathbf{1}_{\mathrm{occupy}(s)},\,0,1\bigr).
$$

Two drives are compared. A weak pulse on `count` stays below $\theta$ and dies. A strong pulse on `count` and `plan` crosses $\theta$; the broadcast then *holds* them. That hold is the schematic of ignition: once in, the same content is available to every module. The labels (`vision`, `lexicon`, `count`, …) are teaching names, not brain regions and not J-lens tokens.

Nothing in the update is a neuron, a residual stream, or a Jacobian. The residual stream of a transformer is a shared write medium; it is not this loop.

In [2]:
result = run_experiment(seed=SEED)
info = facts(result)
for key in (
    "weak_ignited",
    "weak_peak_activity",
    "weak_occupants",
    "strong_ignited",
    "strong_ignition_t",
    "strong_peak_activity",
    "strong_occupants",
):
    print(f"{key:22s} {info[key]}")
print(info["jspace_variance_note"])
fig = render_paper_figure(result)
plt.show()
write_figure(result, root / "paper" / "figures")

weak_ignited           False
weak_peak_activity     0.4056743920844766
weak_occupants         ()
strong_ignited         True
strong_ignition_t      10
strong_peak_activity   1.0
strong_occupants       ('count', 'plan')
Gurnee et al. 2026: J-space component is a small slice (concept-vector median ~6–7%; activation variance never >10%). Not measured here.


/var/folders/1g/_4x26lps6mgg1tr73d7g2bmh0000gp/T/ipykernel_49905/4100092177.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


wrote <checkout>/paper/figures/global_workspace.pdf
wrote <checkout>/paper/figures/global_workspace.png


PosixPath('<checkout>/paper/figures/global_workspace.pdf')

## What to notice

**Ignition is a door, not a dimmer.** Panel C is a step. Weak drive fills zero slots; a little more fills one; the strong trial fills both. There is no valid reading in which “more activity” is “more conscious.” Either the workspace is occupied or it is not.

**Broadcast is what the threshold is for.** In panel B the weak `count` pulse (rust, dashed) is the same module, the same update, a smaller $d_t$. It never enters, so it never gets $\beta$, so it decays. The strong pair crosses $\theta$ and is held at 1. The hold is not extra evidence about the world. It is the architecture writing the winners back.

**Capacity is the other half of access.** Eight modules, two slots. `syntax` and `lexicon` can be busy and still dark. That is the functional claim global workspace theory actually makes: most of what is computed is not poised for report. Panel A is that sentence as a drawing.

**Panel D is a different object.** The $15\times 15$ field is not this toy’s state space. It is a schematic of Gurnee et al. (2026): a small verbalizable slice inside a large automatic volume. Their published numbers — a J-space component around 6–7% of a concept vector, never more than 10% of activation variance, typically $\le 25$ active J-lens vectors — are *theirs*. This notebook did not open Claude.

## A map, not a demonstration: verbalizable representations

[Gurnee et al., *Verbalizable Representations Form a Global Workspace in Language Models*](https://transformer-circuits.pub/2026/workspace) (Transformer Circuits, July 2026) ask a functional question that this toy can only point at. Do language models keep a privileged subset of representations that they can *report on, control, and reason with*, on top of a much larger volume of processing they cannot?

Their answer is yes, for a subspace they call the **J-space**. A *Jacobian lens* (J-lens) finds, at each layer, directions whose first-order causal effect is to make a vocabulary token more sayable — now or later — averaged over many contexts. That is a refinement of the logit lens: the logit lens applies the unembedding as if every layer already spoke the output’s language; the J-lens inserts the average linear map from that layer to the final residual stream first. The tokens that rise in the lens are, in their terms, *verbalizable*: poised to be spoken about if asked, not merely the next token.

They then test the rest of the access cluster, and report that the same directions support:

- **Verbal report.** Asking the model what it is thinking of names J-space content; swapping one J-lens vector for another changes the answer.
- **Directed modulation.** “Hold citrus in mind while copying this sentence” puts `orange` / `lemon` in the lens on an unrelated token.
- **Internal reasoning.** Intermediate values of a silent calculation (`nine`, then `seven`) appear in the workspace and not in the output.
- **Flexible generalisation.** A J-space vector lifted from one context is a valid argument to a different operation in another.
- **Selectivity.** The model can parse, speak fluently, and do a great deal of automatic inference with the J-space suppressed. It struggles at more complex internal reasoning.

A running example in their paper is close to this toy’s `count` module. Prompted to *count to five and introspect*, Claude writes `one` … `five`. The J-lens, on the way, surfaces `counting`, `halfway`, `done`, and a cluster of introspection tokens (`thoughts`, `claude`) that never appear in the output. The surface task and the held thought share a workspace; they are not the same stream.

They are also careful, and the paper follows them. They take **no position** on phenomenal consciousness. Access is a functional notion; its relation to what-it-is-like is the open dispute. They do **not** claim the full Baars / Dehaene architecture. A transformer has no obviously separable input processors, and the broadcast they document is inside one feedforward pass, not a recurrent ignition. They note that competition for the J-space is milder than the sharp ignition this notebook draws. Panel C is the brain-side schematic. Their J-space is a privileged *format* that many circuits can read and write.

Two further results are worth keeping on the map, and not importing into the toy.

**Post-training gives the workspace a point of view.** After assistant training, reactions such as empathy or a safety concern appear in the J-space *while the user message is still being read*. The workspace also carries traces of self-monitoring: `fictional` when role-playing, an internal `BUT` when prefilled against its preferences.

**The same representations can be shaped.** *Counterfactual reflection training* teaches the model what it would say if interrupted and asked to reflect. Behaviour in the uninterrupted context moves, and ablating the implanted J-space concepts (`ethical`, `honest`, `integrity`) largely undoes the move. That is a training consequence of the workspace account: the directions used for verbal report are the ones silent reasoning uses. It is not a method this notebook runs.

None of this is implemented here. There is no Jacobian, no vocabulary, no residual stream. The appropriate use of their paper in this repository is as a *map*: access-like organisation can be asked of a transformer; the answer they give is a small verbalizable slice; that answer is not a consciousness claim.

## Why this example is here

This notebook is not a tutorial on global workspace theory, and it is not an interpretability paper. It is a controlled demonstration of the one fact §7 needs: **most of what is computed is not broadcast; crossing a threshold is a different event from being slightly more active; a privileged, reportable slice is a functional claim that can be asked of a brain and of a model without thereby settling experience.**

The intended readers are the same as the paper’s: clinicians and neuroscientists, and the mathematically inclined who have never met Dehaene’s ignition. The two readings below are of the *same* object. Nothing in the code is a patient, and nothing is a theorem about consciousness.

### What each panel is asked to do

Eight scalars $s\in[0,1]^8$ are updated by the map above. The workspace is a $k$-hot mask, not a separate dynamical system. Ignition time is the first $t$ at which the mask is nonempty. The J-space panel is a $15\times 15$ grid with a $3\times 3$ inset. It does not share a state with A–C.

**For the clinician and the neuroscientist.** You already trust a distinction between what is processed and what can be *said*. A face can be parsed in extra-striate cortex without you being able to report the contour. A word can be primed without entering the conversation. Global workspace theory’s claim is that the difference is architectural: a bottleneck, a competition, a broadcast. Ignition is the moment the lights go on in that shared space — suddenly, not gradually — and the content becomes available for speech, for holding in mind, for a next inference. Panel B is that moment as a threshold. The rust dashed line is the same `count` computation that never makes it; you can be counting as part of a routine and never notice. The teal and gold lines are the count that *does* reach the shared workspace.

Gurnee et al. are the reason this section is no longer only about brains. They find, in a language model, a small set of representations the model can report, hold, and reason with, and a much larger set it cannot. That is the access cluster, not the experience cluster. A system trained to imitate human self-report will produce fluent introspection either way. The J-space result is interesting *because* it is causal and selective, not because the model said it was thinking. Do not take `count` and `plan` as two Brodmann areas, and do not read the teal inset as evidence of experience.

**For the mathematically inclined.** The state is $s_t\in[0,1]^8$. Let $d_t$ be a raised-cosine pulse on one or two coordinates plus a constant baseline. Let $M(s)$ be the $k$-hot mask of coordinates with $s_m\ge\theta$, or $0$ if none qualify. Then

$$
s_{t+1}=\Pi_{[0,1]}\bigl((1-\delta)s_t+d_t+\beta M(s_t)\bigr).
$$

Without the mask the unique fixed point of a constant drive $d$ is $s^\star=d/\delta$ (clipped). The weak trial is built so that $\max_t s_t<\theta$; the strong trial so that two coordinates cross and then stay up because $\beta$ has turned on. Panel C is the map from pulse amplitude to $\|M(s_T)\|_0$. It is a step because $M$ is a step. There is no Lyapunov function and no Hopfield $E$ here; notebook 05 was that story.

The J-space, in Gurnee et al., is a different construction. If $h_\ell$ is the residual stream at layer $\ell$, they estimate

$$
J_\ell=\mathbb{E}_{t,\,t'\ge t,\,\mathrm{prompt}}\Bigl[\frac{\partial h_{\mathrm{final},t'}}{\partial h_{\ell,t}}\Bigr]
$$

and read $\mathrm{softmax}(W_U\,\mathrm{norm}(J_\ell h_\ell))$ as a distribution over tokens. The J-space is then the set of activations expressible as a sparse nonnegative combination of those token-indexed directions (typically $k\le 25$). We do not compute $J_\ell$. Panel D is a cardinality schematic of “small subframe, large complement.”

### What this is not

It is not IIT, not a $\Phi$ estimate, not a recurrent cortical column, not a Jacobian lens, not a logit lens, and not a claim that Claude ignites. The toy has no vocabulary, so it cannot verbalise. Gurnee et al. have a vocabulary and no recurrent workspace loop. The paper’s position is that those are two different facts, and that neither is phenomenal experience.

### What this has to do with being human

Section 6’s landscape was how a system *settles*. This section is how little of what it computes is available to be settled *as a report*. A person who can describe only a sliver of what their visual system just did, who can hold a number in mind while copying a sentence, and who can be asked “what are you thinking?” and name it, is being described — at this grain — as a system with a bottleneck. A foundation model can be described the same way, and Gurnee et al. argue that it should be, without thereby acquiring a body, a cost for being wrong, or an experience of the sliver. The remainder the paper is after sits in that gap: the access we can share as a diagram, and the what-it-is-like we cannot.

Change `SEED` or $\theta$ and the rust pulse and the ignition time will move. The claim should not: **eight modules, two slots, a threshold; most computation stays local; a verbalizable slice is a functional finding, not a consciousness finding.**

### Questions for reflection

Not a quiz. Each question can be asked of a brain and of a machine. Write a short answer that a clinician or a neuroscientist, and a mathematically inclined reader, could both test. The two themes are defined in Further reading below.


1. Most computation is not broadcast ([global workspace](https://en.wikipedia.org/wiki/Global_workspace_theory)). Name a clinical process that is parsed but not reportable, and a model computation that is used but not in a small reportable set.

2. Crossing a threshold is a different event from being slightly more active. Why does a continuous drive plus a step mask give a jump, and why is that not a proof of ignition in cortex?

3. One slot versus two: what can you hold, and what drops? What working-memory fact is that a schematic of?

4. Lesion the winner and a runner may take the workspace. What does that show about “the content,” and what does it not show about experience?

5. A fluent “I am thinking about $X$” can be produced by a system trained to imitate self-report. Why does that fail to settle [phenomenal experience](https://plato.stanford.edu/entries/consciousness/)?

6. Verbalizability is a functional test you can run on a brain and on a model. Why is that theme (i), and why is it not theme (ii)?

7. Raise the threshold on a mid-strength pulse. At what point does the same drive stay local, and what clinical “almost noticed” is that?

8. [IIT](https://en.wikipedia.org/wiki/Integrated_information_theory) is not implemented here. What extra claim would a $\Phi$ estimate or a patient report be making that this teaching model refuses?

9. The rust trace is a count that never enters the workspace. Is unnoticed processing a failure, or the normal case — in a person, and in a model?

10. If a large model has a small reportable set ([mechanistic interpretability](https://en.wikipedia.org/wiki/Mechanistic_interpretability)), have we found a workspace or only a bottleneck in the residual stream?


### Further reading

Each annotation is written against the two themes: **(i) From brains to models** asks what a comparison can establish at a computational level: the same objective, the same representation, the same dynamical description, or the same functional test, applied on both sides — brain and model (machine). **(ii) What AI reveals about being human** asks what the comparison isolates when the systems (human and machine) diverge: [embodiment](https://plato.stanford.edu/entries/embodied-cognition/), a body that must satisfy [allostasis](https://en.wikipedia.org/wiki/Allostasis) (viability) constraints, [phenomenal experience](https://plato.stanford.edu/entries/consciousness/), and values that originate in need and [homeostasis](https://en.wikipedia.org/wiki/Homeostasis) rather than in training data.

The demonstrations in this notebook are teaching models. They illustrate a principle at a scale that can be run and modified. They are not recordings from cortex, not evaluations of a large language model, and not evidence that a next-token probability is a value.

Bibliographic links open the paper or a stable copy. Linked terms open a short definition.

1. **Baars, B. J. (1988).** [*A Cognitive Theory of Consciousness*](https://en.wikipedia.org/wiki/Global_workspace_theory). Cambridge University Press. Entry via the opened GWT page.
   - *(i) From brains to models.* [Global workspace theory](https://en.wikipedia.org/wiki/Global_workspace_theory): specialised processors in parallel; a limited-capacity broadcast when a coalition wins. Baars’s theatre metaphor is a functional architecture (blackboard), not a homunculus. This notebook is that architecture as eight modules and two slots.
   - *(ii) What AI reveals about being human.* GWT is, at its testable core, a theory of [access](https://plato.stanford.edu/entries/consciousness/) — poised for report and control. Dalton’s criticism, recorded on the opened page, is that it does not address what consciousness *is*. The paper agrees: the toy is access, not experience.

2. **Block, N. (1995).** [*On a confusion about a function of consciousness*](https://divinecuration.github.io/assets/pdf/block-confusion.pdf). *Behavioral and Brain Sciences* 18(2), 227–247. Opened PDF.
   - *(i) From brains to models.* [Phenomenal](https://plato.stanford.edu/entries/consciousness/) consciousness is what-it-is-like; [access](https://plato.stanford.edu/entries/consciousness/) consciousness is availability for reasoning, report, and control. The opened paper’s fallacy: an obvious function of access machinery is illicitly transferred to experience (the blindsight example).
   - *(ii) What AI reveals about being human.* This notebook’s ignition step is access. Fluent self-report from a model trained on self-report is also access. Neither is evidence of experience. Keep Block’s distinction in the same sentence as panel C.

3. **Dehaene, S. (2014).** [*Consciousness and the Brain*](https://www.penguinrandomhouse.com/books/310548/consciousness-and-the-brain-by-stanislas-dehaene/). Viking. Publisher page.
   - *(i) From brains to models.* The neuronal workspace: ignition as a nonlinear ignition-and-broadcast, prefrontal and parietal hubs, a threshold. Panel C’s step (not slope) is that claim as a schematic. The opened [GWT](https://en.wikipedia.org/wiki/Global_workspace_theory) page records Dehaene’s avalanche picture and the hard-problem objection.
   - *(ii) What AI reveals about being human.* A jump in occupancy is not a jump in what-it-is-like. The paper’s method is the dissociation biology never gave us: competence, even access-like organisation, without evidence of experience.

4. **Butlin, P., Long, R., et al. (2023).** [*Consciousness in Artificial Intelligence: Insights from the Science of Consciousness*](https://arxiv.org/abs/2308.08708). arXiv:2308.08708.
   - *(i) From brains to models.* Scores current AI against indicator properties drawn from [GWT](https://en.wikipedia.org/wiki/Global_workspace_theory), recurrent processing, higher-order theories, predictive processing, and attention schema. The opened report: no current system is conscious; satisfying more indicators is not an obvious technical impossibility; satisfying them would still not *entail* consciousness.
   - *(ii) What AI reveals about being human.* Use the report as a method, not as a verdict that a future workspace *is* [phenomenal](https://plato.stanford.edu/entries/consciousness/). Theme (ii) is what would still be missing: a theory that predicts experience from ingredients we can add and remove.

5. **Gurnee, W., Sofroniew, N., Pearce, A., et al. (2026).** [*Verbalizable Representations Form a Global Workspace in Language Models*](https://transformer-circuits.pub/2026/workspace). *Transformer Circuits Thread*.
   - *(i) From brains to models.* A Jacobian lens finds a small verbalizable set (J-space) in Claude that supports report, directed modulation, internal reasoning, reuse, and selectivity. Occupancy is a small fraction of activation variance (never more than 10%). The authors take no position on [phenomenal](https://plato.stanford.edu/entries/consciousness/) consciousness and deny that the model reproduces the brain’s workspace: no separable input processors; broadcast in one feedforward pass, not recurrent ignition.
   - *(ii) What AI reveals about being human.* Access-like organisation can emerge without the biology that usually comes with it. That is evidence for theme (i) at the functional grain. It is not evidence that the model is conscious. Panel D is a drawing of their claim, not a measurement of Claude.

6. **Tononi, G., Boly, M., Massimini, M., and Koch, C. (2016).** [*Integrated information theory: from consciousness to its physical substrate*](https://en.wikipedia.org/wiki/Integrated_information_theory). *Nature Reviews Neuroscience* 17, 450–461. Entry via the opened IIT page.
   - *(i) From brains to models.* [IIT](https://en.wikipedia.org/wiki/Integrated_information_theory) identifies experience with a system’s integrated information ($\Phi$), not with a broadcast. It is in the paper because it makes a *divergent* prediction from GWT about which architectures would matter. This toy computes neither $\Phi$ nor a perturbational complexity index.
   - *(ii) What AI reveals about being human.* A high-$\Phi$ claim, if ever made of a model, would still be a theory-laden verdict. Theme (ii) remains open: competence and even integration are not, by themselves, the felt remainder [SEP](https://plato.stanford.edu/entries/consciousness/) calls phenomenal consciousness.

7. **Seth, A. (2021).** [*Being You: A New Science of Consciousness*](https://www.anilseth.com/being-you). Faber & Faber. Author page.
   - *(i) From brains to models.* The view the paper takes to be most defensible: access-like organisation is a functional fact; experience is not that fact. Seth’s emphasis on [allostasis](https://en.wikipedia.org/wiki/Allostasis) and interoception is one candidate for the missing ingredient.
   - *(ii) What AI reveals about being human.* What would change the view is not a more eloquent assistant. It would take a theory that predicts experience from ingredients we can add and remove, and a dissociation that survives the competence we already have.


### Exercises

1. **Raise the threshold.** Ignition is `activity >= THRESHOLD` (default 0.55). Hold a mid-strength pulse (`peak=0.35`, no runner) and re-run at thresholds 0.40, 0.55, and 0.70. Who occupies the workspace, and at which threshold does the same drive stay local?

<!-- solution-box: 06_1_raise_the_threshold.py -->
<details>
<summary><strong>Show solution</strong> — Raise the threshold</summary>

Takes seconds on a laptop. Copy the block below into a new cell. From a clone of the repository, `%load ../solutions/06_1_raise_the_threshold.py` inserts the same code together with the few lines of setup it needs to run outside the notebook.

```python
"""Standalone solution; also %load-able from the companion notebook."""
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

try:
    _SRC = Path(__file__).resolve().parents[1] / "src"
except NameError:
    _cwd = Path.cwd().resolve()
    if (_cwd / "src" / "brainsmodels").is_dir():
        _SRC = _cwd / "src"
    elif (_cwd.parent / "src" / "brainsmodels").is_dir():
        _SRC = _cwd.parent / "src"
    else:
        _SRC = _cwd / "code" / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from brainsmodels.repro import DEFAULT_SEED, set_seed

SEED = set_seed(DEFAULT_SEED)
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

import brainsmodels.workspace as ws
old = ws.THRESHOLD
try:
    for thr in (0.40, 0.55, 0.70):
        ws.THRESHOLD = thr
        trial = ws.run_trial("mid", 0.35, 0.0)
        print(
            f"θ={thr:.2f}  ignited={trial.ignited}  "
            f"t={trial.ignition_t}  occupants={trial.occupants}"
        )
finally:
    ws.THRESHOLD = old
print(
    "\nThe pulse is the same. Crossing a higher bar is a different event "
    "from being slightly more active. Below threshold the computation "
    "stays in the modules; above it, a slot opens. That is the "
    "schematic of ignition, not a measurement of consciousness, and "
    "not Claude."
)
```

</details>
<!-- /solution-box -->


2. **One slot.** Capacity is two. Set `CAPACITY = 1` and re-run the strong trial. Which module is dropped, and what does a one-slot workspace say about “everything that is active is reportable”?

<!-- solution-box: 06_2_one_slot.py -->
<details>
<summary><strong>Show solution</strong> — One slot</summary>

Takes seconds on a laptop. Copy the block below into a new cell. From a clone of the repository, `%load ../solutions/06_2_one_slot.py` inserts the same code together with the few lines of setup it needs to run outside the notebook.

```python
"""Standalone solution; also %load-able from the companion notebook."""
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

try:
    _SRC = Path(__file__).resolve().parents[1] / "src"
except NameError:
    _cwd = Path.cwd().resolve()
    if (_cwd / "src" / "brainsmodels").is_dir():
        _SRC = _cwd / "src"
    elif (_cwd.parent / "src" / "brainsmodels").is_dir():
        _SRC = _cwd.parent / "src"
    else:
        _SRC = _cwd / "code" / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from brainsmodels.repro import DEFAULT_SEED, set_seed

SEED = set_seed(DEFAULT_SEED)
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

import brainsmodels.workspace as ws
old = ws.CAPACITY
try:
    two = ws.run_trial("cap2", ws.STRONG_PEAK, ws.RUNNER_PEAK)
    ws.CAPACITY = 1
    one = ws.run_trial("cap1", ws.STRONG_PEAK, ws.RUNNER_PEAK)
finally:
    ws.CAPACITY = old
print("capacity 2 occupants:", two.occupants)
print("capacity 1 occupants:", one.occupants)
print(
    "\nThe runner-up can be active and still fail to be written back. "
    "Reportability in this toy is occupancy of a scarce buffer, not "
    "strength of a local trace. A verbalizable slice is not phenomenal "
    "experience — that is the §7 limit, and this exercise only makes "
    "the scarcity visible."
)
```

</details>
<!-- /solution-box -->


3. **Lesion the winner.** The intended winner is `count` (module 3). Run a trial whose pulse is only on the runner (`peak=0`, `runner=STRONG_PEAK`). Who ignites? What does a lesion of the would-be winner show about “the workspace chose the meaning”?

<!-- solution-box: 06_3_lesion_the_winner.py -->
<details>
<summary><strong>Show solution</strong> — Lesion the winner</summary>

Takes seconds on a laptop. Copy the block below into a new cell. From a clone of the repository, `%load ../solutions/06_3_lesion_the_winner.py` inserts the same code together with the few lines of setup it needs to run outside the notebook.

```python
"""Standalone solution; also %load-able from the companion notebook."""
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

try:
    _SRC = Path(__file__).resolve().parents[1] / "src"
except NameError:
    _cwd = Path.cwd().resolve()
    if (_cwd / "src" / "brainsmodels").is_dir():
        _SRC = _cwd / "src"
    elif (_cwd.parent / "src" / "brainsmodels").is_dir():
        _SRC = _cwd.parent / "src"
    else:
        _SRC = _cwd / "code" / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from brainsmodels.repro import DEFAULT_SEED, set_seed

SEED = set_seed(DEFAULT_SEED)
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

import brainsmodels.workspace as ws
intact = ws.run_trial("intact", ws.STRONG_PEAK, ws.RUNNER_PEAK)
lesion = ws.run_trial("lesion count", 0.0, ws.STRONG_PEAK)
print("intact :", intact.occupants, "ignition_t", intact.ignition_t)
print("lesion :", lesion.occupants, "ignition_t", lesion.ignition_t)
print(
    "\nWhoever crosses the bar is broadcast. Remove the original winner's "
    "drive and the runner occupies the slots. The workspace did not "
    "interpret the pulse; it admitted the largest local activities. "
    "That is access as a bottleneck, not a mind, and not a clinical "
    "neglect syndrome."
)
```

</details>
<!-- /solution-box -->
